# Peak Detection
This notebook is made for detecting peaks of the snippets extracted in module 2 using the function defined in "3.0 - Functions_peakdetection.ipynb".

Checkout list:
- Going trough all the snippets stored in the data based.
- Applying the peak detection algorithm.
- Storing the detected peaks in a new data file inside the same snippet folder.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import pyarrow.feather as feather

from tqdm import tqdm

from misc.render import *
from misc.synthetic_noise import *
from misc.noise_filtering import *
from misc.peakdetection import *

In [ ]:
#cwd = os.getcwd() # Current Directory
#database_root = os.path.join(cwd,'Databases','Database1') # Local dataset directory

#root = 'data/processed/Database1'
database_root = 'data/processed/Database1' # Local dataset directory

In [ ]:
def write_peaks(path,rate):
    """Function to open snippet files, get peaks, and write them to a new file"""
    df = pd.read_feather(os.path.join(path,'MSNIP.ftr'))
    df_peaks = filtering_det(df['ecg_0'],rate)
    feather.write_feather(df_peaks, os.path.join(path,'PEAKS.ftr'))

In [ ]:
def write_meta(path):
    """Function that creates a new meta file for peak files"""
    data = {'Error Flag': False, 'Error Type': 'No error'}
    with open(os.path.join(path,'PeakMeta.json'), "w") as outfile:
        json.dump(data, outfile)
        outfile.close()

In [ ]:
def find_rate(path):
    """Function which check the meta json file to see the sampling frequency of the original signal"""
    with open(path + '/Meta.json') as json_file:
        metaData = json.load(json_file)
        json_file.close() # Not needed  when you use context
        return int(metaData['Sampling rate'])

In [ ]:
raw_patients = sorted(os.listdir(database_root))

patients = [patient for patient in raw_patients if not patient.endswith(('.json', 'ini','\r'))]
patients = [patient for patient in raw_patients if os.path.isdir(os.path.join(database_root,patient))]
patients_with_holter = [patient for patient in patients if os.path.isdir(os.path.join(database_root,patient,'holter'))]

# Now we need to check the holter folder contains some snippets ---------------------------------------------
# Helper functions (can be offloaded to another file) -------------------------------------------------------


def is_valid_snippet(snippet):
    # We still need to check the expected files are in the folder
    MSNIP.ftr
    
    return True


def has_snippets(path):
    dirs = [a_dir for a_dir in sorted(os.listdir(path)) if os.path.isdir(os.path.join(path,a_dir))]
    snippet_dirs = [a_dir for a_dir in dirs if a_dir.startswith('Snippet')]

    return len(snippet_dirs)>0


def all_snippets(path):
    dirs = [a_dir for a_dir in sorted(os.listdir(path)) if os.path.isdir(os.path.join(path,a_dir))]
    # now we check they are snippet dirs
    snippet_dirs = [a_dir for a_dir in dirs if a_dir.startswith('Snippet')]
    # valid_snippet_dirs = [a_dir for a_dir in snippet_dirs if is_valid_snippet(os.path.join(path,a_dir))]
    return snippet_dirs


# -----------------------------------------------------------------------------------------------------------
    
patients_with_snippets = [patient for patient in patients_with_holter 
                          if has_snippets(os.path.join(database_root,patient,'holter'))]

print(patients_with_snippets)
# Now we have the patient folders with snippets and we know where they are !
# So the loop will be clean

In [ ]:
for patient in tqdm(patients_with_snippets): #Looping throught the folders Sxxxx with snippets
    patient_path = os.path.join(database_root,patient)
    holter_path = os.path.join(patient_path,'holter')
    samplingRate = find_rate(holter_path)
    all_patient_snippets = all_snippets(holter_path)
    all_snippets_path = [os.path.join(holter_path,snippet) for snippet in all_patient_snippets]

    
    print(patient)
    for path_snippet in all_snippets_path:
        write_meta(path_snippet)
        try:
            write_peaks(path_snippet, samplingRate)
        except:
            print("Patient holter file not found, not present in the database.") 

In [ ]:
# Check to see the state of the peaks
for patient in tqdm(patients_with_snippets): #Looping throught the folders Sxxxx with snippets
    patient_path = os.path.join(database_root,patient)
    holter_path = os.path.join(patient_path,'holter')
    samplingRate = find_rate(holter_path)
    all_patient_snippets = all_snippets(holter_path)
    all_snippets_path = [os.path.join(holter_path,snippet) for snippet in all_patient_snippets]

    
    print(patient)
    try:
        for path_snippet in all_snippets_path:
            df = pd.read_feather(path_snippet + '/MSNIP.ftr')
            df_signal = df['ecg_0']
            df_peaks = pd.read_feather(path_snippet + '/PEAKS.ftr')
            plt.plot(df_signal)
            plt.title('ECG signal: ' + patient)
            plt.xlabel('Samples(n)')
            plt.ylabel('Amplitude(mV)')
            plt.scatter(df_peaks['x_values'],df_peaks['y_values'], marker='o',c='red')
            plt.show()
    except:
        print("No holter foder found - moving to the next Snippet.")
        